Dataframe that includes player college production stats, college career average SP ratings

In [31]:
import cfbd
import pandas as pd
import time


configuration = cfbd.Configuration(
    access_token="II7zE93U5v+z5UPByQ1arKsJsU14FUpEkmqMkpKRyzzgrE/kF3a1jtbKNcFEO9P/"
)

START_YEAR = 2006  # to be safe cus want to get all data for players drafted in 2011 and some may have had long college careers
END_YEAR = 2020

# getting team SP ratings
sp_rows = []

with cfbd.ApiClient(configuration) as api_client:
    ratings_api = cfbd.RatingsApi(api_client)
    for year in range(START_YEAR, END_YEAR + 1):
        print(f"Pulling SP+ for {year}...")
        try:
            ratings = ratings_api.get_sp(year=year)
            for team in ratings:
                sp_rows.append({
                    "year": year,
                    "team": team.team,
                    "sp_rating": team.rating
                })
        except Exception as e:
            print(f"Error pulling SP+ for {year}: {e}")
        time.sleep(0.3)

sp_df = pd.DataFrame(sp_rows)

# getting player info
season_rows = []

with cfbd.ApiClient(configuration) as api_client:
    stats_api = cfbd.StatsApi(api_client)
    for year in range(START_YEAR, END_YEAR + 1):
        print(f"Pulling player season stats for {year}...")
        try:
            stats = stats_api.get_player_season_stats(year=year)
            for entry in stats:
                if entry.team is None:
                    continue
                season_rows.append({
                    "player": entry.player,
                    "year": year,
                    "team": entry.team,
                    "college": getattr(entry, "school", None)
                })
        except Exception as e:
            print(f"Error pulling stats for {year}: {e}")
        time.sleep(0.3)

season_df = pd.DataFrame(season_rows)


merged = season_df.merge(
    sp_df,
    on=["year", "team"],
    how="left"
)

merged = merged.dropna(subset=["sp_rating"])


# avg SP for a player during college career
player_avg_sp = (
    merged
    .groupby("player")["sp_rating"]
    .mean()
    .reset_index()
    .rename(columns={"sp_rating": "avg_team_sp"})
)


college_df = pd.read_csv("college_all_positions_clean.csv")

# merging players' career avg SP with their college production data by draft year, round, and pick since unique
player_full = college_df.merge(
    player_avg_sp,
    left_on="Player",  
    right_on="player",
    how="left"
).drop(columns=["player"])

# sorting
player_full_sorted = player_full.sort_values(
    by=["Draft Year", "Round", "Pick"],
    ascending=[True, True, True]
).reset_index(drop=True)


# adding AV and other draft info
draft_info = pd.read_csv("draft_picks.csv")

# columns we want from df
draft_subset = draft_info[["season", "round", "pick", "age", "dr_av", "w_av"]]


full_df = player_full_sorted.merge(
    draft_subset,
    left_on=["Draft Year", "Round", "Pick"],
    right_on=["season", "round", "pick"],
    how="left"
)

full_df = full_df.drop(columns=["season", "round", "pick"])

full_df = full_df.sort_values(
    by=["Draft Year", "Round", "Pick"],
    ascending=[True, True, True]
).reset_index(drop=True)


full_df.to_csv("college_production_with_avg_sp_and_av.csv", index=False)

Pulling SP+ for 2006...
Pulling SP+ for 2007...
Pulling SP+ for 2008...
Pulling SP+ for 2009...
Pulling SP+ for 2010...
Pulling SP+ for 2011...
Pulling SP+ for 2012...
Pulling SP+ for 2013...
Pulling SP+ for 2014...
Pulling SP+ for 2015...
Pulling SP+ for 2016...
Pulling SP+ for 2017...
Pulling SP+ for 2018...
Pulling SP+ for 2019...
Pulling SP+ for 2020...
Pulling player season stats for 2006...
Pulling player season stats for 2007...
Pulling player season stats for 2008...
Pulling player season stats for 2009...
Pulling player season stats for 2010...
Pulling player season stats for 2011...
Pulling player season stats for 2012...
Pulling player season stats for 2013...
Pulling player season stats for 2014...
Pulling player season stats for 2015...
Pulling player season stats for 2016...
Pulling player season stats for 2017...
Pulling player season stats for 2018...
Pulling player season stats for 2019...
Pulling player season stats for 2020...


In [33]:
full_df

,Player,Rate,Draft Team,Round,Pick,Draft Year,Draft College,From,To,G,...,Comb,TFL,Sk/G,Int,IntTD,PD,avg_team_sp,age,dr_av,w_av
0,Cam Newton,178.2,CAR,1,1,2011,Auburn,2007,2010,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,107.0,115.0
1,Von Miller,NaN,DEN,1,2,2011,Texas A&M,2007,2010,47.0,...,181.0,50.5,0.702128,NaN,NaN,NaN,13.800000,22.0,97.0,105.0
2,Marcell Dareus,NaN,BUF,1,3,2011,Alabama,2009,2010,25.0,...,66.0,20.0,0.440000,NaN,NaN,NaN,30.200000,21.0,47.0,56.0
3,A.J. Green,NaN,CIN,1,4,2011,Georgia,2008,2010,32.0,...,NaN,NaN,NaN,NaN,NaN,NaN,16.712000,23.0,71.0,77.0
4,Patrick Peterson,NaN,CRD,1,5,2011,LSU,2008,2010,39.0,...,NaN,NaN,NaN,7.0,1.0,0.0,20.918519,21.0,95.0,103.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2316,Brian Cole,NaN,MIN,7,249,2020,Mississippi State,2018,2019,17.0,...,NaN,NaN,NaN,2.0,0.0,2.0,NaN,23.0,NaN,NaN
2317,Tremayne Anchrum,NaN,RAM,7,250,2020,Clemson,2017,2017,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,26.400000,22.0,2.0,2.0
2318,Stephen Sullivan,NaN,SEA,7,251,2020,LSU,2017,2019,41.0,...,NaN,NaN,NaN,NaN,NaN,NaN,26.568750,23.0,0.0,1.0
2319,Tyrie Cleveland,NaN,DEN,7,252,2020,Florida,2016,2019,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,18.557895,22.0,0.0,0.0
